In [1]:
# Importing the libraries
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn import preprocessing

import sklearn.metrics as metrics
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import pairwise_distances
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer

from surprise.model_selection import cross_validate
from surprise.model_selection import train_test_split

from collections import defaultdict
from surprise import SVD
from surprise import Dataset
from surprise import accuracy
from surprise import Reader


In [2]:
#Reading CSV files
data1 = pd.read_csv('phone_user_review_file_1.csv', encoding=('iso-8859-1'), low_memory= False)
data2 = pd.read_csv('phone_user_review_file_2 (1).csv', encoding=('iso-8859-1'), low_memory= False)
data3 = pd.read_csv('phone_user_review_file_3 (1).csv', encoding=('iso-8859-1'), low_memory= False)
data4 = pd.read_csv('phone_user_review_file_4 (1).csv', encoding=('iso-8859-1'), low_memory= False)
data5 = pd.read_csv('phone_user_review_file_5 (1).csv', encoding=('iso-8859-1'), low_memory= False)
data6 = pd.read_csv('phone_user_review_file_6 (1).csv', encoding=('iso-8859-1'), low_memory= False)

In [3]:
data1.head().T

,0,1,2,3,4
phone_url,/cellphones/samsung-galaxy-s8/,/cellphones/samsung-galaxy-s8/,/cellphones/samsung-galaxy-s8/,/cellphones/samsung-galaxy-s8/,/cellphones/samsung-galaxy-s8/
date,5/2/2017,4/28/2017,5/4/2017,5/2/2017,5/11/2017
lang,en,en,en,en,en
country,us,us,us,us,us
source,Verizon Wireless,Phone Arena,Amazon,Samsung,Verizon Wireless
domain,verizonwireless.com,phonearena.com,amazon.com,samsung.com,verizonwireless.com
score,10.0,10.0,6.0,9.2,4.0
score_max,10.0,10.0,10.0,10.0,10.0
extract,As a diehard Samsung fan who has had every Sam...,Love the phone. the phone is sleek and smooth ...,Adequate feel. Nice heft. Processor's still sl...,Never disappointed. One of the reasons I've be...,I've now found that i'm in a group of people t...
author,CarolAnn35,james0923,R. Craig,Buster2020,S Ate Mine


In [4]:
data1.shape

(374910, 11)

In [5]:
#merging all data into one dataframe
data = pd.concat([data1,data2,data3,data4,data5,data6], ignore_index=True)

In [6]:
data.head()

,phone_url,date,lang,country,source,domain,score,score_max,extract,author,product
0,/cellphones/samsung-galaxy-s8/,5/2/2017,en,us,Verizon Wireless,verizonwireless.com,10.0,10.0,As a diehard Samsung fan who has had every Sam...,CarolAnn35,Samsung Galaxy S8
1,/cellphones/samsung-galaxy-s8/,4/28/2017,en,us,Phone Arena,phonearena.com,10.0,10.0,Love the phone. the phone is sleek and smooth ...,james0923,Samsung Galaxy S8
2,/cellphones/samsung-galaxy-s8/,5/4/2017,en,us,Amazon,amazon.com,6.0,10.0,Adequate feel. Nice heft. Processor's still sl...,R. Craig,"Samsung Galaxy S8 (64GB) G950U 5.8"" 4G LTE Unl..."
3,/cellphones/samsung-galaxy-s8/,5/2/2017,en,us,Samsung,samsung.com,9.2,10.0,Never disappointed. One of the reasons I've be...,Buster2020,Samsung Galaxy S8 64GB (AT&T)
4,/cellphones/samsung-galaxy-s8/,5/11/2017,en,us,Verizon Wireless,verizonwireless.com,4.0,10.0,I've now found that i'm in a group of people t...,S Ate Mine,Samsung Galaxy S8


In [7]:
#Check a few observations and shape of the data-frame
data.shape

(1415133, 11)

In [8]:
data.describe()

,score,score_max
count,1.351644e+06,1351644.0
mean,8.007060e+00,10.0
std,2.616121e+00,0.0
min,2.000000e-01,10.0
25%,7.200000e+00,10.0
50%,9.200000e+00,10.0
75%,1.000000e+01,10.0
max,1.000000e+01,10.0


In [9]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1415133 entries, 0 to 1415132
Data columns (total 11 columns):
 #   Column     Non-Null Count    Dtype  
---  ------     --------------    -----  
 0   phone_url  1415133 non-null  object 
 1   date       1415133 non-null  object 
 2   lang       1415133 non-null  object 
 3   country    1415133 non-null  object 
 4   source     1415133 non-null  object 
 5   domain     1415133 non-null  object 
 6   score      1351644 non-null  float64
 7   score_max  1351644 non-null  float64
 8   extract    1395772 non-null  object 
 9   author     1351931 non-null  object 
 10  product    1415132 non-null  object 
dtypes: float64(2), object(9)
memory usage: 118.8+ MB


In [10]:
#check for missing values
data.isnull().values.any()

True

In [11]:
null_counts = data.isnull().sum()
print (null_counts)

phone_url        0
date             0
lang             0
country          0
source           0
domain           0
score        63489
score_max    63489
extract      19361
author       63202
product          1
dtype: int64


In [12]:
data_copy = data.copy()

In [13]:
#  Impute the missing values if there is any and filling the null values in column 'score' and 'score_max' 
data = data.fillna(data.median())

In [14]:
data = data.dropna() # dropping the null values

In [15]:
# Round oﬀ scores to the nearest integers 
data['score'] = data['score'].astype(int) 
data['score_max'] = data['score_max'].astype(int) 

In [16]:
data.shape

(1336416, 11)

In [17]:
#checking duplicates if any and drop duplicates
data_d = data.drop_duplicates()

In [18]:
# Drop irrelevant features. Keep features like Author, Product, and Score
data_d.drop(['phone_url','date','lang','country','source','domain','score_max','extract'], axis = 1, inplace = True)

In [19]:
data_df = data_d.copy()

In [20]:
data_d.shape

(1331600, 3)

In [21]:
# Keeping only 1000000 data samples, random state=612
df = data_d.sample(n=1000000, random_state=612)

In [22]:
data_d.shape

(1331600, 3)

In [23]:
# Identify the most rated features.
#sorting on products that got highest mean score
df.groupby('product')['score'].mean().sort_values(ascending=False).head()

product
Smartphone Sony Xperia E1 Desbloqueado Vivo Android 4.3 Tela 4 4GB 3G Wi-Fi CÃ¢mera 3MP - Branco                     10.0
Samsung Smartphone Samsung Galaxy S5 Desbloqueado Branco Android 4.4.2 4G CÃ¢mera 16 MP MemÃ³ria Interna 16 GB       10.0
Samsung Smartphone Samsung Galaxy S5 Duos Desbloqueado/ Dual Chip / Branco / 4G / 16 MP / Android 4.4                10.0
Samsung Smartphone Samsung Galaxy S5 Desbloqueado/ Branco / 4G / 16 MP / Android 4.4.2 / 16 GB / USB 3.0             10.0
Samsung Smartphone Samsung Galaxy S5 Desbloqueado Vivo Preto Android 4.4.2 4G CÃ¢mera 16 MP MemÃ³ria Interna 16GB    10.0
Name: score, dtype: float64

In [24]:
# Identify the users with most number of reviews. 
(df['author'].value_counts()).head()

Amazon Customer    57765
Cliente Amazon     14564
e-bit               6309
Client d'Amazon     5720
Amazon Kunde        3624
Name: author, dtype: int64

In [25]:
# The product that got most number of reviews.
df['product'].value_counts().head()

Lenovo Vibe K4 Note (White,16GB)     3908
Lenovo Vibe K4 Note (Black, 16GB)    3234
OnePlus 3 (Graphite, 64 GB)          3128
OnePlus 3 (Soft Gold, 64 GB)         2643
Huawei P8lite zwart / 16 GB          1994
Name: product, dtype: int64

In [26]:
# extracting authors who gave greater than 50 ratings
df1 = pd.DataFrame(columns=['author', 'count'])
df1['author']=df['author'].value_counts().index.tolist() 
df1['a_count'] = list(df['author'].value_counts() > 50)

In [27]:
# get names of indexes for which count column value is False
index_names = df1[ df1['a_count'] == False ].index 

In [28]:
# drop these row indexes from dataFrame 
df1.drop(index_names, inplace = True) 
df1

,author,count,a_count
0,Amazon Customer,NaN,True
1,Cliente Amazon,NaN,True
2,e-bit,NaN,True
3,Client d'Amazon,NaN,True
4,Amazon Kunde,NaN,True
...,...,...,...
674,Ganesh,NaN,True
675,Ralf,NaN,True
676,Jens,NaN,True
677,Roger,NaN,True


In [29]:
# extracting product that got more than 50 ratings
df2 = pd.DataFrame(columns=['product', 'p_count'])
df2['product']=df['product'].value_counts().index.tolist() 
df2['p_count'] = list(df['product'].value_counts() > 50)

In [30]:
# get name_indexes for which count column value is False
index_names = df2[ df2['p_count'] == False ].index 
# drop these row indexes from dataFrame 
df2.drop(index_names, inplace = True)
df2

,product,p_count
0,"Lenovo Vibe K4 Note (White,16GB)",True
1,"Lenovo Vibe K4 Note (Black, 16GB)",True
2,"OnePlus 3 (Graphite, 64 GB)",True
3,"OnePlus 3 (Soft Gold, 64 GB)",True
4,Huawei P8lite zwart / 16 GB,True
...,...,...
4341,Sony Xperia go - Smartphone libre (pantalla tÃ...,True
4342,Sony Ericsson S500i mysterious green (black) H...,True
4343,"LG Electronics L70 Smartphone (11,4 cm (4,5 Zo...",True
4344,Nokia Lumia 530 Smartphone 3G (Ecran: 4 pouces...,True


In [31]:
# selecting data where product is having more than 50 ratings.  
df3 = df[df['product'].isin(df2['product'])] 
df3

,score,author,product
1005326,10,Paul B,Samsung i897 Captivate Android Smartphone Gala...
453603,10,Yuvraj,"Blu Win JR LTE (Grey, 4GB)"
498651,2,Joyce D. Pratt,"BLU Vivo XL Smartphone - 5.5"" 4G LTE - GSM Unl..."
1017703,10,David B,Samsung S3350 Chat 335 Sim Free Mobile Phone
936413,10,Sebastian,"Samsung E1190 Handy (3,6 cm (1,43 Zoll) Displa..."
...,...,...,...
577008,8,Javier,Huawei Ascend Y330 - Smartphone libre Android ...
771460,8,Patrix,"Huawei Ascend G510 Smartphone Touch, Fotocamer..."
600716,2,Amazon Customer,"Apple iPhone 5C Factory Unlocked Cellphone, 8G..."
838993,10,majere1975,"Samsung Smartphone Galaxy S Advance, Display 4..."


In [32]:
df4 = df3[df3['author'].isin(df1['author'])]
df4

,score,author,product
936413,10,Sebastian,"Samsung E1190 Handy (3,6 cm (1,43 Zoll) Displa..."
290678,8,sara,"Samsung SM-N910F Galaxy Note 4 Smartphone, 32 ..."
476314,10,ÐÐ²Ð³ÐµÐ½Ð¸Ð¹,Sony Xperia Z1 Compact (Ð»Ð°Ð¹Ð¼)
223332,8,Amazon Customer,Motorola Moto G 3rd Generation SIM-Free Smartp...
361379,10,e-bit,Smartphone Motorola Moto G 4 Play XT1603
...,...,...,...
396020,2,Amazon customer,Tracfone Motorola Moto E Android Prepaid Phone...
1222820,8,Qantas,Sony Ericsson K810i Cyber-shot
1170633,9,Capyto,Samsung M150 Cep Telefonu
577008,8,Javier,Huawei Ascend Y330 - Smartphone libre Android ...


In [33]:
df4.shape

(108983, 3)

In [34]:
#build a popularity based model

In [36]:
#calculating the mean score for a product by grouping it.
ratings_mean_count = pd.DataFrame(df.groupby('product')['score'].mean()) 

In [37]:
# calculating the number of ratings a product got
ratings_mean_count['rating_counts'] = pd.DataFrame(df.groupby('product')['score'].count())

In [38]:
# recommend top 5 mobile phones
ratings_mean_count.sort_values(by=['score','rating_counts'], ascending=[False,False]).head()

,score,rating_counts
product,,
Samsung Galaxy Note5,10.0,144
Nokia Smartphone Nokia Lumia 520 Desbloqueado Oi Preto Windows Phone 8 CÃ¢mera 5MP 3G Wi-Fi MemÃ³ria Interna 8G GPS,10.0,132
Motorola Smartphone Motorola Moto X Desbloqueado Preto Android 4.2.2 CÃ¢mera 10MP e Frontal 2MP MemÃ³ria Interna de 16GB GSM,10.0,131
Samsung Smartphone Galaxy Win Duos Branco Desbloqueado Dual Chip CÃ¢mera 5MP Processador Quad Core 1.2 Ghz Android 4.1 3G Wi- Fi e MemÃ³ria 8GB,10.0,127
Motorola Smartphone Motorola Moto G Dual Chip Desbloqueado TIM Android 4.3 Tela 4.5 8GB 3G Wi-Fi CÃ¢mera 5MP - Preto,10.0,126


In [39]:
data_pb = df
df

,score,author,product
1005326,10,Paul B,Samsung i897 Captivate Android Smartphone Gala...
453603,10,Yuvraj,"Blu Win JR LTE (Grey, 4GB)"
1010409,10,Pankaj Bhalla,"Lenovo P780 (Deep Black, 4GB)"
866960,6,Bgrazina,Samsung Galaxy XCover 2
498651,2,Joyce D. Pratt,"BLU Vivo XL Smartphone - 5.5"" 4G LTE - GSM Unl..."
...,...,...,...
873202,4,Dudls,Nokia 301 Dual
1267485,8,Cintaaa__,LG Viewty KU990
588916,10,ALBERT M. MASSILLON,BLU Dash JR K Smartphone - Unlocked - Black
102484,2,Amazon Customer,Samsung Galaxy S6 SM-G920F 32GB (FACTORY UNLOC...


In [40]:
df4.groupby('product')['score'].mean().head()  

product
3220                                                                                                                                                                                                      10.000000
5.5-Inch Unlocked Lenovo A850 3G Smartphone-(960x540) Quad Core 4GB MT6582m 1331MHz Android 4.2 Dual Camera +Dual SIM -Black (Rooted + Google Play)                                                        4.500000
AICEK Coque ASUS ZenFone 2 ZE550ML/ZE551ML, AICEK Etui Silicone Gel ASUS ZenFone 2 Housse Antichoc ZenFone 2 Transparente Souple Coque de Protection pour ASUS ZenFone 2(5.5 Pouces)                       9.583333
AICEK Coque ASUS ZenFone 3 ZE520KL, AICEK Etui Silicone Gel ASUS ZenFone 3 Housse Antichoc ZenFone 3 Transparente Souple Coque de Protection pour ASUS ZenFone 3(5.2 Pouces)                               8.421053
AICEK Coque Samsung Galaxy A3 2016, AICEK Etui Silicone Gel Samsung Galaxy A3 2016 (A310F) Housse Antichoc Samsung A3 Transparente Souple Coque 

In [41]:
df4.groupby('product')['score'].mean().sort_values(ascending=False).head()  

product
3220                                                                    10.0
Samsung E1190 SIM free mobile phone                                     10.0
Samsung E1150i Klapphandy 3,6 cm (1,43 Zoll) Display titanium-silver    10.0
Samsung E1150i Klapphandy 3,6 cm (1,43 Zoll) Display ruby-red           10.0
Samsung E1150 Handy (extralange Akkulaufzeit) titanium-silver           10.0
Name: score, dtype: float64

In [42]:
df4.groupby('product')['score'].count().sort_values(ascending=False).head()  


product
Lenovo Vibe K4 Note (White,16GB)       2334
Lenovo Vibe K4 Note (Black, 16GB)      1899
OnePlus 3 (Graphite, 64 GB)            1433
OnePlus 3 (Soft Gold, 64 GB)           1316
Lenovo Vibe K5 (Gold, VoLTE update)    1167
Name: score, dtype: int64

In [43]:
ratings_mean_count = pd.DataFrame(df4.groupby('product')['score'].mean()) 


In [45]:
ratings_mean_count['rating_counts'] = pd.DataFrame(df4.groupby('product')['score'].count())  


In [46]:
ratings_mean_count.head()  

,score,rating_counts
product,,
3220,10.000000,1
5.5-Inch Unlocked Lenovo A850 3G Smartphone-(960x540) Quad Core 4GB MT6582m 1331MHz Android 4.2 Dual Camera +Dual SIM -Black (Rooted + Google Play),4.500000,4
"AICEK Coque ASUS ZenFone 2 ZE550ML/ZE551ML, AICEK Etui Silicone Gel ASUS ZenFone 2 Housse Antichoc ZenFone 2 Transparente Souple Coque de Protection pour ASUS ZenFone 2(5.5 Pouces)",9.583333,24
"AICEK Coque ASUS ZenFone 3 ZE520KL, AICEK Etui Silicone Gel ASUS ZenFone 3 Housse Antichoc ZenFone 3 Transparente Souple Coque de Protection pour ASUS ZenFone 3(5.2 Pouces)",8.421053,19
"AICEK Coque Samsung Galaxy A3 2016, AICEK Etui Silicone Gel Samsung Galaxy A3 2016 (A310F) Housse Antichoc Samsung A3 Transparente Souple Coque De Protection Pour Samsung Galaxy A3 2016 (4,7 pouces)",8.865672,67


#Build a collaborative filtering model using SVD.

In [47]:

# arranging columns in the order of user id,item id and rating to be fed in the svd
columns_titles = ['author','product','score']
df_data = data_df.reindex(columns=columns_titles)

In [48]:
# Keep only 5000 data samples. Use random state=612
sv_data = df_data.sample(n=5000, random_state=612)

In [49]:
# Read dataset.
reader = Reader(rating_scale=(1, 10))
data_U = Dataset.load_from_df(sv_data,reader = reader)

In [54]:
# Use user_based true/false to switch between user-based or item-based collaborative filtering
from surprise import KNNWithMeans
trainset, testset = train_test_split(data_U, test_size=.15)
algo = KNNWithMeans(k=5, sim_options={'name': 'pearson_baseline', 'user_based': True})
algo.fit(trainset)

Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.


In [55]:
# we can now query for specific predicions
uid = 'Frances DeSimone'  # raw user id
iid = 'Samsung Galaxy Star Pro DUOS S7262 Unlocked Ce.'  # raw item id

In [56]:
# get a prediction for specific users and items.
pred = algo.predict(uid, iid, verbose=True)

user: Frances DeSimone item: Samsung Galaxy Star Pro DUOS S7262 Unlocked Ce. r_ui = None   est = 8.00   {'was_impossible': True, 'reason': 'User and/or item is unknown.'}


In [57]:
# run the trained model against the testset
test_pred = algo.test(testset)

In [58]:
test_pred

[Prediction(uid='mikhet', iid='Sony Ericsson Xperia X10 Mini Pro', r_ui=5.0, est=8.002823529411765, details={'was_impossible': True, 'reason': 'User and/or item is unknown.'}),
 Prediction(uid='Rajat', iid='Samsung Galaxy J7 SM-J700F (Black, 16GB)', r_ui=6.0, est=8.002823529411765, details={'was_impossible': True, 'reason': 'User and/or item is unknown.'}),
 Prediction(uid='ksinbln', iid='ALCATEL ONETOUCH 1010D-2AALDE1 One Touch Tiger Handy (3,7 cm (1,5 Zoll) Display, 4MB RAM, Dual-Band GSM, micro-USB) schwarz', r_ui=6.0, est=8.002823529411765, details={'was_impossible': True, 'reason': 'User and/or item is unknown.'}),
 Prediction(uid='Amazon Customer', iid='Motorola Moto G 3rd Generation (Black, 8GB)', r_ui=8.0, est=10, details={'actual_k': 1, 'was_impossible': False}),
 Prediction(uid='Vincent', iid='Huawei P8lite zwart / 16 GB', r_ui=8.0, est=8.002823529411765, details={'was_impossible': True, 'reason': 'User and/or item is unknown.'}),
 Prediction(uid='ertanonkok', iid='BlackBerry

In [59]:
# get RMSE
print("User-based Model : Test Set")
accuracy.rmse(test_pred, verbose=True)

User-based Model : Test Set
RMSE: 2.5506


2.550607094700349

In [62]:
reader = Reader(rating_scale=(1, 10))
data_I = Dataset.load_from_df(sv_data,reader = reader)
trainset2, testset2 = train_test_split(data_I, test_size=.15)

In [63]:
# Use user_based true/false to switch between user-based or item-based collaborative filtering
algo = KNNWithMeans(k=5, sim_options={'name': 'pearson_baseline', 'user_based': False})
algo.fit(trainset)

Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.


In [64]:
# run the trained model against the testset
test_pred = algo.test(testset)

In [65]:
test_pred

[Prediction(uid='mikhet', iid='Sony Ericsson Xperia X10 Mini Pro', r_ui=5.0, est=8.002823529411765, details={'was_impossible': True, 'reason': 'User and/or item is unknown.'}),
 Prediction(uid='Rajat', iid='Samsung Galaxy J7 SM-J700F (Black, 16GB)', r_ui=6.0, est=8.002823529411765, details={'was_impossible': True, 'reason': 'User and/or item is unknown.'}),
 Prediction(uid='ksinbln', iid='ALCATEL ONETOUCH 1010D-2AALDE1 One Touch Tiger Handy (3,7 cm (1,5 Zoll) Display, 4MB RAM, Dual-Band GSM, micro-USB) schwarz', r_ui=6.0, est=8.002823529411765, details={'was_impossible': True, 'reason': 'User and/or item is unknown.'}),
 Prediction(uid='Amazon Customer', iid='Motorola Moto G 3rd Generation (Black, 8GB)', r_ui=8.0, est=10, details={'actual_k': 5, 'was_impossible': False}),
 Prediction(uid='Vincent', iid='Huawei P8lite zwart / 16 GB', r_ui=8.0, est=8.002823529411765, details={'was_impossible': True, 'reason': 'User and/or item is unknown.'}),
 Prediction(uid='ertanonkok', iid='BlackBerry

In [66]:
# get RMSE
print("Item-based Model : Test Set")
accuracy.rmse(test_pred, verbose=True)

Item-based Model : Test Set
RMSE: 2.5736


2.5736258142234063

In [67]:
#  Try cross validation techniques to get better results.
cross_validate(algo,data_U, measures=['RMSE'], cv=3, verbose=False)

Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.


{'test_rmse': array([2.60908946, 2.66919232, 2.63918779]),
 'fit_time': (0.6337075233459473, 0.4591193199157715, 0.6717054843902588),
 'test_time': (0.032320261001586914, 0.05779743194580078, 0.05498552322387695)}

##In what business scenario you should use popularity based Recommendation Systems ?
Ans- As the name suggests Popularity based recommendation system works with the trend or frequent used. It basically uses the items which are in trend right now. For example, if any product which is usually bought by every new user then there are chances that it may suggest that item to the user who just signed up.

The best example of this system is Google News

It is used by the travel companies selling holiday packages in a season, by Google News and other news websites to show Top Stories with images.

##In what business scenario you should use Collaborative Filtering based Recommendation Systems ?
Ans-Collaborative filtering systems make recommendations based on historic users’ preference for items (clicked, watched, purchased, liked, rated, etc.). The preference can be presented as a user-item matrix. 

Most websites like Amazon, YouTube, and Netflix use collaborative filtering as a part of their sophisticated recommendation system

##What other possible methods can you think of which can further improve the recommendation for diﬀerent users ?
Ans- Apart from Popularity and Collaborative Filtering , Content-based, Demographic, Utility based, Knowledge based and Hybrid recommendation system can be used as per the user needs.